# Airbnb Review Tier Prediction
Cleaned portfolio notebook. Place the Montreal Airbnb `listings.csv` file in `../data/` before running.


## 1) Intro & problem
Predict listing review-quality tiers for Montreal Airbnb listings so new or unreviewed listings can be ranked with a quality signal.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast
import string
import unicodedata
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score, recall_score, accuracy_score, precision_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix


## 2) Data loading / cleaning
Load the listings data, remove metadata/text fields, drop missing target rows, remove extreme max-night outliers, and convert strings to typed fields.


In [ ]:
df = pd.read_csv('../data/listings.csv')
df.head()

In [ ]:
df_clean = df.copy().drop(columns=[
    'id',
    'listing_url',
    'scrape_id',
    'name',
    'license',
    'last_scraped',
    'source',
    'picture_url',
    'host_id',
    'host_url',
    'host_name',
    'host_thumbnail_url',
    'host_picture_url',
    'calendar_last_scraped',
    'description',
    'neighborhood_overview',
    'host_about',
    'neighbourhood'
])

In [ ]:
#host_neighbourhood is mostly empty, take it out to preserve data
# Can safely drop bc the reviews will be "cheating" for data
df_cleaner_no_null = df_cleaner.drop(columns = ['host_response_time',
                                                'host_response_rate',
                                                'host_neighbourhood',
                                                'number_of_reviews',
                                                'number_of_reviews_ltm',
                                                'number_of_reviews_l30d',
                                                'review_scores_accuracy',
                                                'review_scores_cleanliness',
                                                'review_scores_checkin',
                                                'review_scores_communication',
                                                'review_scores_location',
                                                'review_scores_value',
                                                'reviews_per_month',
                                                'first_review',
                                                'last_review',
                                                'number_of_reviews_ly',
                                                'host_is_superhost', # Saves 300 rows
                                                #'host_response_rate', # Saves 100 rows
                                                #'host_acceptance_rate',
                                                'host_verifications' # bc almost unary, not very useful
                                                ])
#fill
df_cleaner_no_null['host_location'] = df_cleaner_no_null['host_location'].fillna('Unknown')


# Can safely drop bc if no rating not useful for train

df_cleaner_no_null = df_cleaner_no_null[df_cleaner_no_null['review_scores_rating'].notna()]  


df_cleaner_no_null = df_cleaner_no_null.dropna()
# Everything left seems potentially too useful to drop

df_cleaner_no_null.info()

In [ ]:
df_even_cleaner = df_cleaner_no_null.copy().drop(columns= ['has_availability'])

# Outlier. Troll ?
df_even_cleaner = df_even_cleaner.loc[((df_even_cleaner['maximum_nights'] < 10000) &
                                       (df_even_cleaner['minimum_maximum_nights'] < 10000) &
                                       (df_even_cleaner['maximum_maximum_nights'] < 10000) &
                                       (df_even_cleaner['maximum_nights_avg_ntm'] < 10000)
                                       ), :]

# Datetime
df_even_cleaner['host_since'] = pd.to_datetime(df_even_cleaner['host_since'])
#df_even_cleaner['first_review'] = pd.to_datetime(df_even_cleaner['first_review'])
#df_even_cleaner['last_review'] = pd.to_datetime(df_even_cleaner['last_review'])

# Str -> numbers

#df_even_cleaner['host_response_rate'] = (df_even_cleaner['host_response_rate'].str.rstrip("%").astype(float)/100)
df_even_cleaner['host_acceptance_rate'] = (df_even_cleaner['host_acceptance_rate'].str.rstrip("%").astype(float)/100)
df_even_cleaner['price'] = df_even_cleaner['price'].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)

## 3) Feature engineering & leakage control
Remove review-history fields unavailable for new listings, lump categories, engineer property and amenity indicators, and create rating tiers.


In [ ]:
def host_location_mapper(location):
    if any(x in location for x in ['Montreal', 'Côte Saint-Luc', 'Pointe-Claire', 'Beaconsfield']):
        return 'Montreal'
    elif any(x in location for x in ['Laval', 'Châteauguay', 'Westmount', 'Longueuil', 'Québec City', 'Repentigny', 'Sherbrooke', 'Terrebonne']):
        return 'Quebec'
    elif 'Canada' in location:
        return 'Canada'
    elif 'Unknown' in location:
        return 'Unknown'
    else:
        return 'Other'
    

def min_lump(x, value_counts):
    if x in value_counts:
        return x
    else:
        return 'Other'
    

def keyword(description, keyword):
    punctuation_to_remove = string.punctuation + '’‘'
    
    if isinstance(description, list):
        description = " ".join(description)
    
    description_clean = description.translate(str.maketrans('', '', punctuation_to_remove)).lower()
    keyword_clean = keyword.translate(str.maketrans('', '', punctuation_to_remove)).lower()
    
    return keyword_clean in description_clean

In [ ]:
df_even_cleaner['host_location_clean'] = df_even_cleaner['host_location'].apply(lambda x: host_location_mapper(x))

In [ ]:
df_even_cleaner['neighbourhood_cleansed_lump'] = df_even_cleaner['neighbourhood_cleansed'].apply(lambda x: min_lump(x, df_even_cleaner['neighbourhood_cleansed'].value_counts()[0:20]))

In [ ]:
df_even_cleaner['Condo'] = df_even_cleaner['property_type'].apply(lambda x: keyword(x, 'condo'))
df_even_cleaner['Rental'] = df_even_cleaner['property_type'].apply(lambda x: keyword(x, 'rental unit'))
df_even_cleaner['Townhouse'] = df_even_cleaner['property_type'].apply(lambda x: keyword(x, 'townhouse'))
df_even_cleaner['Home'] = df_even_cleaner['property_type'].apply(lambda x: keyword(x, 'home'))
df_even_cleaner['Loft'] = df_even_cleaner['property_type'].apply(lambda x: keyword(x, 'loft'))
df_even_cleaner['Apartment'] = df_even_cleaner['property_type'].apply(lambda x: keyword(x, 'apartment'))
df_even_cleaner['Hotel'] = df_even_cleaner['property_type'].apply(lambda x: keyword(x, 'hotel'))
df_even_cleaner['Hostel'] = df_even_cleaner['property_type'].apply(lambda x: keyword(x, 'hostel'))
df_even_cleaner['Suite'] = df_even_cleaner['property_type'].apply(lambda x: keyword(x, 'suite'))

In [ ]:
df_even_cleaner['Shared_bath'] = df_even_cleaner['bathrooms_text'].apply(lambda x: keyword(x, 'shared'))
df_even_cleaner['Private_bath'] = df_even_cleaner['bathrooms_text'].apply(lambda x: keyword(x, 'private'))

In [ ]:
df_even_cleaner['amenities_2'] = df_even_cleaner['amenities'].apply(ast.literal_eval)

In [ ]:
df_copy = df_even_cleaner.copy()

def remove_punct(text):
    if isinstance(text, str):
        all_punct = string.punctuation + "’"
        return text.translate(str.maketrans('', '', all_punct)).strip()
    return text

top_amenities = (
    df_even_cleaner['amenities']
    .apply(ast.literal_eval)              
    .apply(lambda lst: [remove_punct(i) for i in lst])  
    .explode()                           
    .value_counts()[:100]                 
)

def check_amenity_in_description(description, amenity):
    return keyword(description, amenity)

for amenity in top_amenities.index: 
    amenity_str = str(amenity)
    column_name = f'has_{amenity_str.replace(" ", "_").lower()}'
    df_copy[column_name] = df_copy['amenities_2'].apply(lambda x: check_amenity_in_description(x, amenity_str))


In [ ]:
df_new = df_copy.drop(columns = [
    'amenities',
    'amenities_2',
    'bathrooms_text',
    'neighbourhood_cleansed',
    'property_type',
    'host_location',
])

In [ ]:
df_final = df_new.drop(columns = ['calculated_host_listings_count',
                                  'host_listings_count',
                                  'minimum_minimum_nights',
                                  'maximum_minimum_nights',
                                  'minimum_maximum_nights',
                                  'maximum_maximum_nights',
                                  'availability_60'
                                  ])

## Redundant or very high correl

In [ ]:
string_columns = []

for col in df_final.columns:
    if df_final[col].dtype == object:
        string_columns.append(col)
        print(f"{col}: \n{df_final[col].unique()}")


dummy = pd.get_dummies(df_final[string_columns], drop_first=True, dtype=int)
dummed_df = pd.concat([df_final.drop(columns=string_columns), dummy], axis=1)

In [ ]:
dummed_df['host_since_time'] = dummed_df['host_since'].astype(int) / 10**9 
dummed_df_2 = dummed_df.drop(columns = 'host_since')

In [ ]:
### I'm going to try changing it from regression to classification -
# we need to know which ones to recommend but it doesn't super matter like a 1 vs 1.2, both bad
# three groups: 4.8-5 good 4.5-4.8 ok and below bad
# (based on online comments)

In [ ]:
dummed_df['rating_cat'] = dummed_df['review_scores_rating'].apply(
    lambda x: 3 if x >= 4.8 else (2 if x >= 4.5 else 1)
)
dummed_df_2 = dummed_df.drop(columns = ['host_since', 'review_scores_rating'])

In [ ]:
dummed_df_2['rating_cat'].value_counts()/len(dummed_df_2['rating_cat'])

In [ ]:
y = dummed_df_2['rating_cat']
X = dummed_df_2.drop(columns=['rating_cat'])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## 4) Model training & comparison
Compare ANN, L1-regularized logistic workflow, and random forest approaches using macro F1.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', MLPClassifier(
        max_iter=1000,
        random_state=42,
        early_stopping = True
    ))
])

param_grid = {
'model__hidden_layer_sizes': [(5,) ,(10,), (20,), (40,), (10, 10)],
'model__learning_rate_init': [ 0.001, 0.01]
}

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring='f1_macro',   
    cv=kf, verbose = 3
)

grid.fit(X_train, y_train)

 
print(grid.best_score_)
print(grid.best_params_)

y_test_pred = grid.predict(X_test)
final_recall = f1_score(y_test, y_test_pred, average='macro')
print(final_recall)

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', MLPClassifier(hidden_layer_sizes=(40, ),
                          max_iter=1000,
                          random_state=42,
                           learning_rate_init = 0.01,
        early_stopping = True))
])

pipeline.fit(X_train, y_train)

# Predict
y_test_pred = pipeline.predict(X_test)

recall = f1_score(y_test, y_test_pred, average='macro')
print(recall)

In [ ]:
parameters = {'C': [1/0.1, 1/0.01, 1/0.001, 1/0.0001]}

log = LogisticRegression(random_state = 42, penalty='l1', solver='saga', max_iter = 10000)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
grid_log = GridSearchCV(log, parameters, cv=kf, scoring = 'f1_macro')

grid_log.fit(X_train, y_train)

print(grid_log.best_score_) 
print(grid_log.best_params_)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectFromModel(LogisticRegression(penalty='l1', solver='saga', C=10))),
    ('model', MLPClassifier(
        hidden_layer_sizes=(40, ),
        learning_rate_init=0.01,
        max_iter=1000,
        random_state=42,
        early_stopping = True
    ))
])

pipe.fit(X_train, y_train)

y_test_pred = pipe.predict(X_test)
recall = f1_score(y_test, y_test_pred, average='macro')
print(recall)

lasso_model = pipe.named_steps['feature_selection'].estimator_

coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lasso_model.coef_[0]
})

coef_df['abs_coefficient'] = coef_df['coefficient'].abs()

coef_df = coef_df.sort_values(by='abs_coefficient', ascending=False)
coef_df = coef_df.drop(columns='abs_coefficient')

 
coef_df_notdropped = coef_df[coef_df['coefficient'] != 0]
print(coef_df_notdropped)
print(len(coef_df_notdropped))


coef_df_dropped = coef_df[coef_df['coefficient'] == 0]
print(coef_df_dropped)
print(len(coef_df_dropped))



In [ ]:
parameters = {'n_estimators':[50,100,150],
              'max_features': [3,4,5],
              'min_samples_leaf':[1,2,3,4,5],
             'max_depth': [3,5,7,9,11]}

rf = RandomForestClassifier(random_state = 42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
grid_rf = GridSearchCV(rf, parameters, cv=kf, scoring = 'f1_macro')

grid_rf.fit(X_train, y_train)

print(grid_rf.best_score_) 
print(grid_rf.best_params_)

In [ ]:
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection',SelectFromModel(RandomForestClassifier(random_state=42,
                                                                n_estimators=150,
                                                                max_features=5,
                                                                min_samples_leaf=1,
                                                                max_depth=11))),
    ('model', MLPClassifier(
        hidden_layer_sizes=(40,),
        learning_rate_init=0.01,
        max_iter=1000,
        random_state=42,
        early_stopping = True
    ))
])
pipe_rf.fit(X_train, y_train)

y_test_pred = pipe_rf.predict(X_test)
recall = f1_score(y_test, y_test_pred, average='macro')

print(recall)

rf_model = pipe_rf.named_steps['feature_selection'].estimator_

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
})

 
selector_rf = pipe_rf.named_steps['feature_selection']
support = selector_rf.get_support()

importance_df_notdropped = importance_df[support]
print(importance_df_notdropped)
print(len(importance_df_notdropped))

importance_df_dropped = importance_df[~support]
print(importance_df_dropped)
print(len(importance_df_dropped))

## 5) Evaluation
Evaluate macro F1, class distribution, predictions, and confusion matrices.


In [ ]:
confusion_matrix(y_test,
                 y_test_pred)

In [ ]:
confusion_matrix(y_test,
                 y_test_pred,
                 normalize='true') #So worse at guessing first two

In [ ]:
np.unique(y_test, return_counts=True)

In [ ]:
np.unique(y_test_pred, return_counts=True)

## 6) Interpretation / business takeaways
The report selected the interpretable L1/logistic workflow and linked stronger review tiers to convenience and host-experience signals. Class imbalance remains the main limitation.
